# Étude de cas 3 — Jalon 3 : du Holter au patch

**Objectif :** adapter un classifieur ECG appris sur Holter A à Patch B, qui a peu de patients annotés et un signal enregistré différemment. Les expériences sur le patch seront comparées par validation croisée groupée par patient.

**Protocole :** pour l’instant, ce notebook ne charge que train et validation. Le test est réservé à l’évaluation finale du modèle retenu. Les données sont simulées et n’ont pas de portée médicale.

**Parcours du Jalon :** (1) établir le résultat du CNN Holter appliqué tel quel au patch ; (2) entraîner sur le peu de données patch ; (3) comparer extraction de caractéristiques et ajustement fin ; (4) essayer l’augmentation patch et la combinaison avec l’ajustement fin ; (5) ajouter la prédiction de fréquence cardiaque comme tâche auxiliaire ; (6) rédiger la décision et les limites. Chaque partie sera ajoutée et expliquée avant la suivante.


## 1. Vérifier les bibliothèques

Cette cellule affiche les versions de Python et des bibliothèques qui servent à charger les données, entraîner le CNN et mesurer ses prédictions.


In [3]:
from importlib.metadata import version, PackageNotFoundError
import sys
print('Python :', sys.executable)
for paquet in ['numpy', 'scikit-learn', 'tensorflow', 'matplotlib']:
    try:
        print(f'{paquet} : {version(paquet)}')
    except PackageNotFoundError:
        print(f'{paquet} : ABSENT')


Python : /home/mapster/ABOUBAKER/SDIA_PRACTICES/SCHOOL/s3-s4/advanced-ai/USECASE_TP/PRACTICE/.venv/bin/python
numpy : 2.5.3
scikit-learn : 1.9.1
tensorflow : 2.21.0
matplotlib : 3.11.2


## 2. Charger uniquement les données train et validation

Holter A apporte les données servant à apprendre le modèle de départ. Patch B fournit les 12 patients train utilisés dans la validation croisée groupée, ainsi que ses 6 patients validation. On ne charge pas les fenêtres test.


In [5]:
from pathlib import Path
import numpy as np

# Trouver la racine PRACTICE si le notebook est ouvert depuis son sous-dossier notebooks.
candidats = [Path.cwd(), *Path.cwd().parents]
PRACTICE_DIR = next((p for p in candidats if (p / 'data' / 'holter_A').is_dir()), None)
if PRACTICE_DIR is None:
    raise FileNotFoundError('Ouvrir ce notebook depuis PRACTICE ou un de ses sous-dossiers.')
DELIVERABLE_DIR = PRACTICE_DIR / 'deliverables' / 'Jalon3'
DELIVERABLE_DIR.mkdir(parents=True, exist_ok=True)

# Ne lire que les clés train/val afin de garder les tests fermés.
with np.load(PRACTICE_DIR / 'data' / 'holter_A' / 'uc3_holterA.npz') as f:
    A = {k: f[k].copy() for k in ['X_train','classe_train','patient_train','X_val','classe_val','patient_val',
                                  'fc_train','fc_val','age_train','sexe_train']}
with np.load(PRACTICE_DIR / 'data' / 'patch_B' / 'uc3_patchB.npz') as f:
    B = {k: f[k].copy() for k in ['X_train','classe_train','patient_train','X_val','classe_val','patient_val',
                                  'fc_train','fc_val','age_train','sexe_train']}

for nom, D in [('Holter A', A), ('Patch B', B)]:
    print(f'{nom}: train {D["X_train"].shape}, val {D["X_val"].shape}')
    print('  patients train/val :', len(np.unique(D['patient_train'])), '/', len(np.unique(D['patient_val'])))
    print('  classes train :', dict(zip(*np.unique(D['classe_train'], return_counts=True))))
    print('  classes val :', dict(zip(*np.unique(D['classe_val'], return_counts=True))))
    assert not (set(np.unique(D['patient_train'])) & set(np.unique(D['patient_val'])))
print('Vérification : aucun patient partagé entre train et validation dans chaque appareil.')


Holter A: train (8400, 1250), val (1800, 1250)
  patients train/val : 280 / 60
  classes train : {np.int8(0): np.int64(4348), np.int8(1): np.int64(1209), np.int8(2): np.int64(1018), np.int8(3): np.int64(995), np.int8(4): np.int64(830)}
  classes val : {np.int8(0): np.int64(892), np.int8(1): np.int64(238), np.int8(2): np.int64(292), np.int8(3): np.int64(198), np.int8(4): np.int64(180)}
Patch B: train (300, 1250), val (150, 1250)
  patients train/val : 12 / 6
  classes train : {np.int8(0): np.int64(119), np.int8(1): np.int64(69), np.int8(2): np.int64(46), np.int8(3): np.int64(29), np.int8(4): np.int64(37)}
  classes val : {np.int8(0): np.int64(67), np.int8(1): np.int64(6), np.int8(2): np.int64(43), np.int8(3): np.int64(9), np.int8(4): np.int64(25)}
Vérification : aucun patient partagé entre train et validation dans chaque appareil.


## 3. Normaliser chaque fenêtre

On soustrait la moyenne de chaque fenêtre et on divise par son écart-type. Cela retire un décalage constant et compense un gain constant. Le résultat a trois dimensions : fenêtres, 1 250 mesures, un canal ECG.


In [7]:
FS = 125

def normaliser_fenetres(X):
    # Normaliser dans le temps, séparément pour chaque fenêtre.
    X = X.astype('float32')
    mu = X.mean(axis=1, keepdims=True)
    sd = X.std(axis=1, keepdims=True)
    return ((X - mu) / (sd + 1e-6))[..., None]

SA = {s: normaliser_fenetres(A['X_' + s]) for s in ['train', 'val']}
SB = {s: normaliser_fenetres(B['X_' + s]) for s in ['train', 'val']}
print('Forme Holter train :', SA['train'].shape)
print('Forme patch train :', SB['train'].shape)
print('Moyenne/écart-type d’une fenêtre normalisée :',
      round(float(SA['train'][0].mean()), 3), round(float(SA['train'][0].std()), 3))


Forme Holter train : (8400, 1250, 1)
Forme patch train : (300, 1250, 1)
Moyenne/écart-type d’une fenêtre normalisée : 0.0 1.0


## 4. Créer les plis patch sans séparer les fenêtres d’un même patient

Il y a 12 patients patch train. À chaque tour, 9 patients servent à entraîner ou régler la méthode et 3 patients sont gardés pour la validation du pli. Les quatre tours couvrent les 12 patients. Cette cellule prépare les indices ; elle ne fait encore aucune prédiction.


In [9]:
from sklearn.model_selection import GroupKFold

N_PLIS = 4
groupes = B['patient_train']
plis_patch = list(GroupKFold(n_splits=N_PLIS).split(SB['train'], B['classe_train'], groups=groupes))
for numero, (idx_train, idx_val) in enumerate(plis_patch, start=1):
    print(f'Pli {numero}: {len(np.unique(groupes[idx_train]))} patients entraînement, '
          f'{len(np.unique(groupes[idx_val]))} validation ; '
          f'{len(idx_train)} / {len(idx_val)} fenêtres')
    assert not (set(np.unique(groupes[idx_train])) & set(np.unique(groupes[idx_val])))


Pli 1: 9 patients entraînement, 3 validation ; 225 / 75 fenêtres
Pli 2: 9 patients entraînement, 3 validation ; 225 / 75 fenêtres
Pli 3: 9 patients entraînement, 3 validation ; 225 / 75 fenêtres
Pli 4: 9 patients entraînement, 3 validation ; 225 / 75 fenêtres


## 5. Construire le CNN de départ fourni pour Jalon 3

Le CNN apprend d’abord sur Holter A. Quatre blocs `convolution + max-pooling` repèrent des motifs locaux, puis une couche dense nommée `representation` forme un résumé interne. La dernière couche transforme ce résumé en probabilités pour les cinq rythmes. Le nom `representation` sera utile plus tard pour essayer l’extraction de caractéristiques.


In [11]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, accuracy_score, recall_score, confusion_matrix

GRAINE = 42
keras.utils.set_random_seed(GRAINE)
CLASSES = np.arange(5)

def cnn1d():
    entree = keras.Input((1250, 1), name='ecg')
    x = entree
    for filtres in (16, 32, 64, 64):
        x = layers.Conv1D(filtres, 7, padding='same', activation='relu')(x)
        x = layers.MaxPooling1D(2)(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64, activation='relu', name='representation')(x)
    rythme = layers.Dense(5, activation='softmax', name='rythme')(x)
    return keras.Model(entree, rythme, name='cnn1d_holter')

poids = compute_class_weight('balanced', classes=CLASSES, y=A['classe_train'])
POIDS = dict(zip(CLASSES.tolist(), poids.tolist()))
print('Poids des classes Holter train :', {k: round(v, 3) for k,v in POIDS.items()})


Poids des classes Holter train : {0: 0.386, 1: 1.39, 2: 1.65, 3: 1.688, 4: 2.024}


/home/mapster/.config/matplotlib is not a writable directory
Matplotlib created a temporary cache directory at /tmp/matplotlib-wpqqre35 because there was an issue with the default path (/home/mapster/.config/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


## 6. Apprendre le modèle de référence sur Holter A

C’est le modèle « tel quel » : il apprend sur Holter train et ne voit aucune fenêtre Patch B pendant l’entraînement. Holter validation sert à surveiller l’apprentissage ; l’expérience d’adaptation sera évaluée sur les plis patch.


In [13]:
cnn_A = cnn1d()
cnn_A.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
historique_A = cnn_A.fit(SA['train'], A['classe_train'],
                         class_weight=POIDS,
                         epochs=12, batch_size=64,
                         validation_data=(SA['val'], A['classe_val']),
                         verbose=2)


Epoch 1/12
132/132 - 10s - 79ms/step - accuracy: 0.4436 - loss: 1.1078 - val_accuracy: 0.5406 - val_loss: 0.8257
Epoch 2/12
132/132 - 8s - 62ms/step - accuracy: 0.7424 - loss: 0.4700 - val_accuracy: 0.8867 - val_loss: 0.3225
Epoch 3/12
132/132 - 8s - 63ms/step - accuracy: 0.8785 - loss: 0.2406 - val_accuracy: 0.9167 - val_loss: 0.2087
Epoch 4/12
132/132 - 9s - 66ms/step - accuracy: 0.9220 - loss: 0.1619 - val_accuracy: 0.9322 - val_loss: 0.1689
Epoch 5/12
132/132 - 9s - 66ms/step - accuracy: 0.9401 - loss: 0.1296 - val_accuracy: 0.9522 - val_loss: 0.1315
Epoch 6/12
132/132 - 8s - 62ms/step - accuracy: 0.9488 - loss: 0.1108 - val_accuracy: 0.9633 - val_loss: 0.1021
Epoch 7/12
132/132 - 8s - 61ms/step - accuracy: 0.9493 - loss: 0.1064 - val_accuracy: 0.9628 - val_loss: 0.1042
Epoch 8/12
132/132 - 8s - 63ms/step - accuracy: 0.9576 - loss: 0.0872 - val_accuracy: 0.9711 - val_loss: 0.0759
Epoch 9/12
132/132 - 8s - 60ms/step - accuracy: 0.9625 - loss: 0.0784 - val_accuracy: 0.9706 - val_loss

## 7. Première comparaison : CNN Holter appliqué au patch sans adaptation

Le modèle reste inchangé. On prédit les fenêtres des patients gardés hors du pli, puis on rassemble les prédictions hors-pli des quatre tours. Le F1 macro donne le même poids aux cinq rythmes ; le rappel par classe montre lesquels sont trouvés.


In [15]:
# Chaque fenêtre patch reçoit une prédiction uniquement lorsqu’elle est dans un pli tenu à l’écart.
pred_patch_oof = np.full(len(B['classe_train']), -1, dtype=int)
proba_patch_oof = np.full((len(B['classe_train']), 5), np.nan, dtype='float32')
for numero, (idx_train, idx_val) in enumerate(plis_patch, start=1):
    # Ici le modèle n’est pas ajusté au patch : on mesure seulement le transfert Holter -> patch.
    proba = cnn_A.predict(SB['train'][idx_val], verbose=0)
    proba_patch_oof[idx_val] = proba
    pred_patch_oof[idx_val] = proba.argmax(axis=1)
    print(f'Pli {numero} terminé : {len(idx_val)} fenêtres patch prédites.')

F1_PATCH_ASIS = f1_score(B['classe_train'], pred_patch_oof, average='macro', zero_division=0)
ACC_PATCH_ASIS = accuracy_score(B['classe_train'], pred_patch_oof)
RAPPELS_PATCH_ASIS = recall_score(B['classe_train'], pred_patch_oof, labels=CLASSES,
                                  average=None, zero_division=0)
print('Holter CNN appliqué au patch, CV groupée par patient :')
print('Exactitude :', round(ACC_PATCH_ASIS, 3))
print('F1 macro :', round(F1_PATCH_ASIS, 3))
print('Rappel classes 0 à 4 :', np.round(RAPPELS_PATCH_ASIS, 3))
print('Matrice (lignes vraies, colonnes prédites) :\n',
      confusion_matrix(B['classe_train'], pred_patch_oof, labels=CLASSES))


Pli 1 terminé : 75 fenêtres patch prédites.
Pli 2 terminé : 75 fenêtres patch prédites.
Pli 3 terminé : 75 fenêtres patch prédites.
Pli 4 terminé : 75 fenêtres patch prédites.
Holter CNN appliqué au patch, CV groupée par patient :
Exactitude : 0.6
F1 macro : 0.621
Rappel classes 0 à 4 : [0.395 0.565 1.    0.379 1.   ]
Matrice (lignes vraies, colonnes prédites) :
 [[47  2 10  0 60]
 [ 0 39  2  0 28]
 [ 0  0 46  0  0]
 [ 0  4  0 11 14]
 [ 0  0  0  0 37]]


## 8. Sauvegarder le modèle de référence et consigner le premier résultat

Ce modèle servira de point de comparaison pour les prochaines stratégies. On sauvegarde l’artefact et le score hors-pli, sans consulter les jeux test.


In [17]:
from pathlib import Path

FICHIER_CNN_HOLTER = DELIVERABLE_DIR / 'modele_cnn_holter_jalon3.keras'
cnn_A.save(FICHIER_CNN_HOLTER)

import csv
JOURNAL = DELIVERABLE_DIR / 'journal_experiences_jalon3.csv'
champs = ['jalon','experience','graine','f1_macro_validation','exactitude_validation',
          'rappel_classes_0_a_4','statut','notes']
with JOURNAL.open('w', newline='', encoding='utf-8-sig') as f:
    ecrivain = csv.DictWriter(f, fieldnames=champs); ecrivain.writeheader()
    ecrivain.writerow({'jalon':3, 'experience':'CNN_Holter_applique_patch_sans_adaptation',
        'graine':GRAINE, 'f1_macro_validation':round(float(F1_PATCH_ASIS),4),
        'exactitude_validation':round(float(ACC_PATCH_ASIS),4),
        'rappel_classes_0_a_4':' / '.join(f'{v:.3f}' for v in RAPPELS_PATCH_ASIS),
        'statut':'réalisé',
        'notes':'prédictions hors-pli sur les 12 patients patch train ; GroupKFold par patient ; aucun test chargé'})
print('CNN de référence sauvegardé :', FICHIER_CNN_HOLTER)
print('Journal créé :', JOURNAL)


CNN de référence sauvegardé : /home/mapster/ABOUBAKER/SDIA_PRACTICES/SCHOOL/s3-s4/advanced-ai/USECASE_TP/PRACTICE/deliverables/Jalon3/modele_cnn_holter_jalon3.keras
Journal créé : /home/mapster/ABOUBAKER/SDIA_PRACTICES/SCHOOL/s3-s4/advanced-ai/USECASE_TP/PRACTICE/deliverables/Jalon3/journal_experiences_jalon3.csv


## 9. Deuxième comparaison : entraîner un CNN uniquement sur Patch B

Cette fois, chaque pli crée un CNN neuf, sans poids appris sur Holter. Il s’entraîne sur les 9 patients Patch B du pli et prédit les 3 patients tenus à l’écart. On garde la même architecture et le même nombre d’époques que pour le modèle Holter pour comparer le point de départ.


In [19]:
# Prédictions hors-pli : chaque patient est évalué par un modèle qui ne l’a pas vu.
y_patch = B['classe_train'].astype(int)
pred_patch_only_oof = np.full(len(y_patch), -1, dtype=int)

for numero, (idx_train, idx_val) in enumerate(plis_patch, start=1):
    keras.backend.clear_session()
    keras.utils.set_random_seed(GRAINE + numero)
    modele_patch = cnn1d()  # poids neufs : aucun apprentissage préalable sur Holter
    modele_patch.compile(optimizer=keras.optimizers.Adam(1e-3),
                         loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    # Calculer les poids de classe avec les seuls patients d’entraînement de ce pli.
    classes_vues = np.unique(y_patch[idx_train])
    poids_pli = compute_class_weight('balanced', classes=classes_vues, y=y_patch[idx_train])
    poids_pli = dict(zip(classes_vues.tolist(), poids_pli.tolist()))

    modele_patch.fit(SB['train'][idx_train], y_patch[idx_train],
                     class_weight=poids_pli, epochs=12, batch_size=64, verbose=0)
    proba = modele_patch(SB['train'][idx_val], training=False).numpy()
    pred_patch_only_oof[idx_val] = proba.argmax(axis=1)
    f1_pli = f1_score(y_patch[idx_val], pred_patch_only_oof[idx_val],
                      average='macro', zero_division=0)
    print(f'Pli {numero} : F1 macro={f1_pli:.3f} sur {len(np.unique(B["patient_train"][idx_val]))} patients tenus à l’écart.')
    del modele_patch

assert np.all(pred_patch_only_oof >= 0), 'Certaines fenêtres n’ont pas reçu de prédiction hors-pli.'
F1_PATCH_ONLY = f1_score(y_patch, pred_patch_only_oof, average='macro', zero_division=0)
ACC_PATCH_ONLY = accuracy_score(y_patch, pred_patch_only_oof)
RAPPELS_PATCH_ONLY = recall_score(y_patch, pred_patch_only_oof, labels=CLASSES,
                                  average=None, zero_division=0)
print('CNN entraîné seulement sur les 12 patients patch :')
print('Exactitude CV :', round(ACC_PATCH_ONLY, 3))
print('F1 macro CV :', round(F1_PATCH_ONLY, 3))
print('Rappel classes 0 à 4 :', np.round(RAPPELS_PATCH_ONLY, 3))
print('Écart de F1 par rapport au CNN Holter sans adaptation :',
      round(float(F1_PATCH_ONLY - F1_PATCH_ASIS), 3))

# Ajouter cet essai au journal créé dans la section précédente.
with JOURNAL.open('a', newline='', encoding='utf-8-sig') as f:
    ecrivain = csv.DictWriter(f, fieldnames=champs)
    ecrivain.writerow({'jalon':3, 'experience':'CNN_entraine_sur_patch_seul',
        'graine':GRAINE, 'f1_macro_validation':round(float(F1_PATCH_ONLY),4),
        'exactitude_validation':round(float(ACC_PATCH_ONLY),4),
        'rappel_classes_0_a_4':' / '.join(f'{v:.3f}' for v in RAPPELS_PATCH_ONLY),
        'statut':'réalisé',
        'notes':'validation croisée 4 plis groupés par patient ; initialisation aléatoire ; poids de classe calculés par pli'})
print('Journal mis à jour :', JOURNAL)


Pli 1 : F1 macro=0.389 sur 3 patients tenus à l’écart.
Pli 2 : F1 macro=0.175 sur 3 patients tenus à l’écart.
Pli 3 : F1 macro=0.103 sur 3 patients tenus à l’écart.
Pli 4 : F1 macro=0.231 sur 3 patients tenus à l’écart.
CNN entraîné seulement sur les 12 patients patch :
Exactitude CV : 0.36
F1 macro CV : 0.344
Rappel classes 0 à 4 : [0.395 0.246 0.63  0.448 0.054]
Écart de F1 par rapport au CNN Holter sans adaptation : -0.277
Journal mis à jour : /home/mapster/ABOUBAKER/SDIA_PRACTICES/SCHOOL/s3-s4/advanced-ai/USECASE_TP/PRACTICE/deliverables/Jalon3/journal_experiences_jalon3.csv


## 10. Première interprétation

Comparer les deux F1 macro hors-pli : le premier modèle sait déjà des choses apprises sur Holter ; le second apprend à partir des patients patch, mais dispose de très peu de données. Le score seul ne suffit pas : lire aussi les rappels par classe et leur variabilité entre plis. Ces résultats portent uniquement sur les 12 patients patch d’entraînement en validation croisée, pas sur le test final.


In [21]:
print(f'Holter sans adaptation : F1 macro={F1_PATCH_ASIS:.3f}')
print(f'CNN patch seul : F1 macro={F1_PATCH_ONLY:.3f}')
print('Patch seul - Holter direct :', round(float(F1_PATCH_ONLY-F1_PATCH_ASIS),3))


Holter sans adaptation : F1 macro=0.621
CNN patch seul : F1 macro=0.344
Patch seul - Holter direct : -0.277


## 11. Troisième comparaison : extraction de caractéristiques

On reprend le CNN entraîné sur Holter. Ses couches convolutionnelles et sa représentation dense restent gelées : elles continuent à repérer les motifs appris sur Holter. Seule la dernière couche, qui choisit parmi les cinq rythmes, est ajustée avec les patients patch du pli. C’est comme garder le détecteur de motifs et réapprendre seulement la décision finale.


In [23]:
import csv

# Prédictions patch hors-pli après ajustement de la seule couche de sortie.
pred_extraction_oof = np.full(len(B['classe_train']), -1, dtype=int)
N_PARAMETRES_EXTRACTION = None

for numero, (idx_train, idx_val) in enumerate(plis_patch, start=1):
    keras.utils.set_random_seed(GRAINE + 100 + numero)

    # Dupliquer le CNN Holter et recopier exactement ses poids appris.
    modele_extraction = keras.models.clone_model(cnn_A)
    modele_extraction.set_weights(cnn_A.get_weights())

    # Tout reste fixe sauf la sortie « rythme » : seuls ces poids apprennent avec Patch B.
    for couche in modele_extraction.layers:
        couche.trainable = (couche.name == 'rythme')
    modele_extraction.compile(optimizer=keras.optimizers.Adam(1e-3),
                              loss='sparse_categorical_crossentropy')

    nombre_trainable = int(sum(np.prod(w.shape) for w in modele_extraction.trainable_weights))
    if N_PARAMETRES_EXTRACTION is None:
        N_PARAMETRES_EXTRACTION = nombre_trainable
    assert nombre_trainable == N_PARAMETRES_EXTRACTION

    # Les poids d’équilibrage n’utilisent que les étiquettes des patients train du pli.
    y_train_pli = B['classe_train'][idx_train].astype(int)
    classes_vues = np.unique(y_train_pli)
    poids_pli = compute_class_weight('balanced', classes=classes_vues, y=y_train_pli)
    poids_pli = dict(zip(classes_vues.tolist(), poids_pli.tolist()))

    modele_extraction.fit(SB['train'][idx_train], y_train_pli, class_weight=poids_pli,
                          epochs=12, batch_size=64, verbose=0)
    # Appel direct pour obtenir les scores du pli sans entraîner sur ses patients tenus à l’écart.
    proba = modele_extraction(SB['train'][idx_val], training=False).numpy()
    pred_extraction_oof[idx_val] = proba.argmax(axis=1)
    score_pli = f1_score(B['classe_train'][idx_val], pred_extraction_oof[idx_val],
                         average='macro', zero_division=0)
    print(f'Pli {numero} : F1 macro={score_pli:.3f} ; paramètres entraînables={nombre_trainable}.')
    del modele_extraction

assert np.all(pred_extraction_oof >= 0)
F1_EXTRACTION = f1_score(B['classe_train'], pred_extraction_oof, average='macro', zero_division=0)
ACC_EXTRACTION = accuracy_score(B['classe_train'], pred_extraction_oof)
RAPPELS_EXTRACTION = recall_score(B['classe_train'], pred_extraction_oof, labels=CLASSES,
                                  average=None, zero_division=0)
print('Extraction de caractéristiques — Patch B, CV patient :')
print('Exactitude :', round(ACC_EXTRACTION, 3))
print('F1 macro :', round(F1_EXTRACTION, 3))
print('Rappel classes 0 à 4 :', np.round(RAPPELS_EXTRACTION, 3))
print('Paramètres entraînables :', N_PARAMETRES_EXTRACTION)

with JOURNAL.open('a', newline='', encoding='utf-8-sig') as f:
    ecrivain = csv.DictWriter(f, fieldnames=champs)
    ecrivain.writerow({'jalon':3, 'experience':'extraction_caracteristiques_sortie_seule',
        'graine':GRAINE, 'f1_macro_validation':round(float(F1_EXTRACTION),4),
        'exactitude_validation':round(float(ACC_EXTRACTION),4),
        'rappel_classes_0_a_4':' / '.join(f'{v:.3f}' for v in RAPPELS_EXTRACTION),
        'statut':'réalisé',
        'notes':f'CV 4 plis groupés par patient ; base Holter gelée sauf couche de sortie ; {N_PARAMETRES_EXTRACTION} paramètres entraînables'})
print('Journal mis à jour :', JOURNAL)


Pli 1 : F1 macro=0.653 ; paramètres entraînables=325.
Pli 2 : F1 macro=0.664 ; paramètres entraînables=325.
Pli 3 : F1 macro=0.845 ; paramètres entraînables=325.
Pli 4 : F1 macro=0.464 ; paramètres entraînables=325.
Extraction de caractéristiques — Patch B, CV patient :
Exactitude : 0.78
F1 macro : 0.771
Rappel classes 0 à 4 : [0.748 0.812 0.935 0.655 0.73 ]
Paramètres entraînables : 325
Journal mis à jour : /home/mapster/ABOUBAKER/SDIA_PRACTICES/SCHOOL/s3-s4/advanced-ai/USECASE_TP/PRACTICE/deliverables/Jalon3/journal_experiences_jalon3.csv


## 12. Interpréter l’extraction de caractéristiques

Comparer son score au CNN Holter inchangé et au CNN patch seul. Ici, le détecteur de motifs reste appris sur Holter ; seule la décision entre les cinq classes a pu s’ajuster aux rares patients patch. La prochaine expérience laissera aussi des couches internes s’adapter : l’ajustement fin.


In [25]:
print(f'Holter sans adaptation : F1 macro={F1_PATCH_ASIS:.3f}')
print(f'Patch seul : F1 macro={F1_PATCH_ONLY:.3f}')
print(f'Extraction de caractéristiques : F1 macro={F1_EXTRACTION:.3f}')
print('Paramètres entraînables avec extraction :', N_PARAMETRES_EXTRACTION)


Holter sans adaptation : F1 macro=0.621
Patch seul : F1 macro=0.344
Extraction de caractéristiques : F1 macro=0.771
Paramètres entraînables avec extraction : 325


## 13. Quatrième comparaison : ajustement fin du CNN Holter

On repart encore des poids Holter, mais cette fois toutes les couches sont dégelées : les motifs internes et la décision finale peuvent s’adapter au patch. On utilise un taux d’apprentissage plus petit pour modifier les poids progressivement. Avec seulement 9 patients par pli, ce modèle a aussi plus de possibilités de trop s’adapter aux exemples vus.


In [27]:
# Prédictions hors-pli du CNN Holter entièrement ajusté sur les patients patch du pli.
pred_finetune_oof = np.full(len(B['classe_train']), -1, dtype=int)
N_PARAMETRES_FINETUNE = None

for numero, (idx_train, idx_val) in enumerate(plis_patch, start=1):
    keras.utils.set_random_seed(GRAINE + 200 + numero)

    # Chaque pli repart du même CNN appris sur Holter, pas du pli précédent.
    modele_finetune = keras.models.clone_model(cnn_A)
    modele_finetune.set_weights(cnn_A.get_weights())
    for couche in modele_finetune.layers:
        couche.trainable = True

    nombre_trainable = int(sum(np.prod(w.shape) for w in modele_finetune.trainable_weights))
    if N_PARAMETRES_FINETUNE is None:
        N_PARAMETRES_FINETUNE = nombre_trainable
    assert nombre_trainable == N_PARAMETRES_FINETUNE

    modele_finetune.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-4),
                             loss='sparse_categorical_crossentropy')
    y_train_pli = B['classe_train'][idx_train].astype(int)
    classes_vues = np.unique(y_train_pli)
    poids_pli = compute_class_weight('balanced', classes=classes_vues, y=y_train_pli)
    poids_pli = dict(zip(classes_vues.tolist(), poids_pli.tolist()))

    modele_finetune.fit(SB['train'][idx_train], y_train_pli, class_weight=poids_pli,
                        epochs=12, batch_size=64, verbose=0)
    proba = modele_finetune(SB['train'][idx_val], training=False).numpy()
    pred_finetune_oof[idx_val] = proba.argmax(axis=1)
    score_pli = f1_score(B['classe_train'][idx_val], pred_finetune_oof[idx_val],
                         average='macro', zero_division=0)
    print(f'Pli {numero} : F1 macro={score_pli:.3f} ; paramètres entraînables={nombre_trainable}.')
    del modele_finetune

assert np.all(pred_finetune_oof >= 0)
F1_FINETUNE = f1_score(B['classe_train'], pred_finetune_oof, average='macro', zero_division=0)
ACC_FINETUNE = accuracy_score(B['classe_train'], pred_finetune_oof)
RAPPELS_FINETUNE = recall_score(B['classe_train'], pred_finetune_oof, labels=CLASSES,
                                average=None, zero_division=0)
print('Ajustement fin complet — Patch B, CV patient :')
print('Exactitude :', round(ACC_FINETUNE, 3))
print('F1 macro :', round(F1_FINETUNE, 3))
print('Rappel classes 0 à 4 :', np.round(RAPPELS_FINETUNE, 3))
print('Paramètres entraînables :', N_PARAMETRES_FINETUNE)

with JOURNAL.open('a', newline='', encoding='utf-8-sig') as f:
    ecrivain = csv.DictWriter(f, fieldnames=champs)
    ecrivain.writerow({'jalon':3, 'experience':'ajustement_fin_complet',
        'graine':GRAINE, 'f1_macro_validation':round(float(F1_FINETUNE),4),
        'exactitude_validation':round(float(ACC_FINETUNE),4),
        'rappel_classes_0_a_4':' / '.join(f'{v:.3f}' for v in RAPPELS_FINETUNE),
        'statut':'réalisé',
        'notes':f'CV 4 plis groupés par patient ; CNN Holter entièrement dégelé ; taux apprentissage 1e-4 ; {N_PARAMETRES_FINETUNE} paramètres entraînables'})
print('Journal mis à jour :', JOURNAL)


Pli 1 : F1 macro=0.777 ; paramètres entraînables=51365.
Pli 2 : F1 macro=0.647 ; paramètres entraînables=51365.
Pli 3 : F1 macro=0.853 ; paramètres entraînables=51365.
Pli 4 : F1 macro=0.497 ; paramètres entraînables=51365.
Ajustement fin complet — Patch B, CV patient :
Exactitude : 0.823
F1 macro : 0.823
Rappel classes 0 à 4 : [0.739 0.87  0.935 0.862 0.838]
Paramètres entraînables : 51365
Journal mis à jour : /home/mapster/ABOUBAKER/SDIA_PRACTICES/SCHOOL/s3-s4/advanced-ai/USECASE_TP/PRACTICE/deliverables/Jalon3/journal_experiences_jalon3.csv


## 14. Comparer les stratégies déjà testées

Ces quatre expériences réutilisent la même séparation par patient. L’extraction adapte seulement la sortie ; l’ajustement fin complet peut modifier toutes les couches. Ici, l’ajustement fin obtient le meilleur F1 macro provisoire, mais les 12 patients et les écarts entre plis imposent de rester prudent. Aucun score test n’est encore calculé.

In [29]:
print(f'Holter appliqué sans adaptation : F1 macro={F1_PATCH_ASIS:.3f}')
print(f'CNN patch seul : F1 macro={F1_PATCH_ONLY:.3f}')
print(f'Extraction de caractéristiques : F1 macro={F1_EXTRACTION:.3f} ({N_PARAMETRES_EXTRACTION} paramètres entraînables)')
print(f'Ajustement fin complet : F1 macro={F1_FINETUNE:.3f} ({N_PARAMETRES_FINETUNE} paramètres entraînables)')


Holter appliqué sans adaptation : F1 macro=0.621
CNN patch seul : F1 macro=0.344
Extraction de caractéristiques : F1 macro=0.771 (325 paramètres entraînables)
Ajustement fin complet : F1 macro=0.823 (51365 paramètres entraînables)


## 15. Entraîner le CNN Holter avec une augmentation de type patch

Le code de départ propose quatre perturbations en mV : gain aléatoire, dérive lente, bruit secteur à 50 Hz et bruit musculaire. On les ajoute au signal brut Holter, puis on normalise chaque fenêtre. Pour garder la même taille d’entraînement que le CNN de référence, environ la moitié des fenêtres Holter est perturbée et l’autre reste propre. Le modèle ne voit aucune donnée Patch B pendant cet entraînement.

In [31]:
# Créer des versions Holter perturbées avec les niveaux proposés dans le code de départ.
def bruit_patch_j3(X, graine):
    rng = np.random.default_rng(graine)
    X = X.astype('float32')
    n, longueur = X.shape
    temps = np.arange(longueur, dtype='float32') / FS

    # Variation de gain et dérive lente de la ligne de base.
    gain = rng.uniform(0.5, 1.0, (n, 1)).astype('float32')
    frequence_derive = rng.uniform(0.1, 0.5, (n, 1)).astype('float32')
    derive = 0.1 * np.sin(2 * np.pi * frequence_derive * temps[None, :])

    # Bruit électrique à 50 Hz et bruit musculaire aléatoire.
    secteur = 0.02 * np.sin(2 * np.pi * 50 * temps)[None, :]
    muscle = rng.normal(0.0, 0.03, X.shape).astype('float32')
    return (gain * X + derive + secteur + muscle).astype('float32')

X_holter_aug_brut = bruit_patch_j3(A['X_train'], GRAINE + 300)
# Perturber aléatoirement la moitié des fenêtres, sans doubler la taille du jeu.
rng_melange = np.random.default_rng(GRAINE + 301)
masque_aug = rng_melange.random(len(A['X_train'])) < 0.5
X_holter_mixte_brut = A['X_train'].copy()
X_holter_mixte_brut[masque_aug] = X_holter_aug_brut[masque_aug]
y_holter_mixte = A['classe_train'].astype(int)
X_holter_mixte = normaliser_fenetres(X_holter_mixte_brut)
print('Fenêtres Holter augmentées :', int(masque_aug.sum()), '/', len(masque_aug))

keras.utils.set_random_seed(GRAINE + 300)
cnn_A_aug = cnn1d()
cnn_A_aug.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss='sparse_categorical_crossentropy')
cnn_A_aug.fit(X_holter_mixte, y_holter_mixte, class_weight=POIDS,
              epochs=12, batch_size=64,
              validation_data=(SA['val'], A['classe_val']), verbose=0)

# Ce modèle n'a encore vu aucune donnée Patch B pendant son apprentissage.
pred_aug_oof = cnn_A_aug(SB['train'], training=False).numpy().argmax(axis=1)
F1_AUG = f1_score(B['classe_train'], pred_aug_oof, average='macro', zero_division=0)
ACC_AUG = accuracy_score(B['classe_train'], pred_aug_oof)
RAPPELS_AUG = recall_score(B['classe_train'], pred_aug_oof, labels=CLASSES,
                           average=None, zero_division=0)
for numero, (_, idx_val) in enumerate(plis_patch, start=1):
    score_pli = f1_score(B['classe_train'][idx_val], pred_aug_oof[idx_val],
                         average='macro', zero_division=0)
    print(f'Pli {numero} : F1 macro={score_pli:.3f}.')
print('Holter entraîné avec augmentation patch, puis appliqué au patch :')
print('Exactitude :', round(ACC_AUG, 3))
print('F1 macro :', round(F1_AUG, 3))
print('Rappel classes 0 à 4 :', np.round(RAPPELS_AUG, 3))

# Inscrire ou remplacer cette expérience pour éviter les lignes dupliquées en cas de réexécution.
experience_aug = {'jalon':3, 'experience':'Holter_avec_augmentation_patch',
    'graine':GRAINE, 'f1_macro_validation':round(float(F1_AUG),4),
    'exactitude_validation':round(float(ACC_AUG),4),
    'rappel_classes_0_a_4':' / '.join(f'{v:.3f}' for v in RAPPELS_AUG),
    'statut':'réalisé',
    'notes':'Holter propre + une version augmentée par fenêtre ; transfert évalué par plis patient patch ; aucun patient patch utilisé pour entraîner'}
with JOURNAL.open(encoding='utf-8-sig', newline='') as f:
    lignes = list(csv.DictReader(f))
lignes = [r for r in lignes if r['experience'] != experience_aug['experience']]
lignes.append(experience_aug)
with JOURNAL.open('w', encoding='utf-8-sig', newline='') as f:
    ecrivain = csv.DictWriter(f, fieldnames=champs); ecrivain.writeheader(); ecrivain.writerows(lignes)
cnn_A_aug.save(DELIVERABLE_DIR / 'modele_cnn_holter_augmente_jalon3.keras')
print('Modèle et journal sauvegardés dans :', DELIVERABLE_DIR)

Fenêtres Holter augmentées : 4216 / 8400
Pli 1 : F1 macro=0.401.
Pli 2 : F1 macro=0.641.
Pli 3 : F1 macro=0.727.
Pli 4 : F1 macro=0.376.
Holter entraîné avec augmentation patch, puis appliqué au patch :
Exactitude : 0.703
F1 macro : 0.687
Rappel classes 0 à 4 : [0.622 0.623 0.978 0.448 0.973]
Modèle et journal sauvegardés dans : /home/mapster/ABOUBAKER/SDIA_PRACTICES/SCHOOL/s3-s4/advanced-ai/USECASE_TP/PRACTICE/deliverables/Jalon3


## 16. Combiner l’augmentation Holter et l’ajustement fin

On reprend maintenant le CNN entraîné sur Holter augmenté. Pour chaque pli, on repart des mêmes poids Holter augmentés, puis on ajuste toutes les couches avec les 9 patients patch du pli. Les 3 patients tenus à l’écart ne participent pas à cet ajustement.

In [33]:
# Réentraîner séparément un modèle à chaque pli en partant des poids Holter augmentés.
pred_aug_finetune_oof = np.full(len(B['classe_train']), -1, dtype=int)
N_PARAMETRES_AUG_FINETUNE = None

for numero, (idx_train, idx_val) in enumerate(plis_patch, start=1):
    keras.utils.set_random_seed(GRAINE + 300 + numero)
    modele_aug_finetune = keras.models.clone_model(cnn_A_aug)
    modele_aug_finetune.set_weights(cnn_A_aug.get_weights())
    for couche in modele_aug_finetune.layers:
        couche.trainable = True

    nombre_trainable = int(sum(np.prod(w.shape) for w in modele_aug_finetune.trainable_weights))
    N_PARAMETRES_AUG_FINETUNE = nombre_trainable
    modele_aug_finetune.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-4),
                                loss='sparse_categorical_crossentropy')
    y_train_pli = B['classe_train'][idx_train].astype(int)
    classes_vues = np.unique(y_train_pli)
    poids_pli = compute_class_weight('balanced', classes=classes_vues, y=y_train_pli)
    poids_pli = dict(zip(classes_vues.tolist(), poids_pli.tolist()))
    modele_aug_finetune.fit(SB['train'][idx_train], y_train_pli, class_weight=poids_pli,
                            epochs=12, batch_size=64, verbose=0)
    proba = modele_aug_finetune(SB['train'][idx_val], training=False).numpy()
    pred_aug_finetune_oof[idx_val] = proba.argmax(axis=1)
    score_pli = f1_score(B['classe_train'][idx_val], pred_aug_finetune_oof[idx_val],
                         average='macro', zero_division=0)
    print(f'Pli {numero} : F1 macro={score_pli:.3f} ; paramètres entraînables={nombre_trainable}.')
    del modele_aug_finetune

assert np.all(pred_aug_finetune_oof >= 0)
F1_AUG_FINETUNE = f1_score(B['classe_train'], pred_aug_finetune_oof,
                           average='macro', zero_division=0)
ACC_AUG_FINETUNE = accuracy_score(B['classe_train'], pred_aug_finetune_oof)
RAPPELS_AUG_FINETUNE = recall_score(B['classe_train'], pred_aug_finetune_oof, labels=CLASSES,
                                    average=None, zero_division=0)
print('Augmentation Holter + ajustement fin — CV patient Patch B :')
print('Exactitude :', round(ACC_AUG_FINETUNE, 3))
print('F1 macro :', round(F1_AUG_FINETUNE, 3))
print('Rappel classes 0 à 4 :', np.round(RAPPELS_AUG_FINETUNE, 3))
print('Paramètres entraînables pendant le fine-tuning :', N_PARAMETRES_AUG_FINETUNE)

experience_combo = {'jalon':3, 'experience':'augmentation_plus_ajustement_fin',
    'graine':GRAINE, 'f1_macro_validation':round(float(F1_AUG_FINETUNE),4),
    'exactitude_validation':round(float(ACC_AUG_FINETUNE),4),
    'rappel_classes_0_a_4':' / '.join(f'{v:.3f}' for v in RAPPELS_AUG_FINETUNE),
    'statut':'réalisé',
    'notes':f'CV 4 plis par patient ; CNN Holter préentraîné avec augmentation puis entièrement ajusté sur Patch B ; {N_PARAMETRES_AUG_FINETUNE} paramètres entraînables'}
with JOURNAL.open(encoding='utf-8-sig', newline='') as f:
    lignes = list(csv.DictReader(f))
lignes = [r for r in lignes if r['experience'] != experience_combo['experience']]
lignes.append(experience_combo)
with JOURNAL.open('w', encoding='utf-8-sig', newline='') as f:
    ecrivain = csv.DictWriter(f, fieldnames=champs); ecrivain.writeheader(); ecrivain.writerows(lignes)
print('Journal mis à jour :', JOURNAL)

Pli 1 : F1 macro=0.741 ; paramètres entraînables=51365.
Pli 2 : F1 macro=0.724 ; paramètres entraînables=51365.
Pli 3 : F1 macro=0.836 ; paramètres entraînables=51365.
Pli 4 : F1 macro=0.511 ; paramètres entraînables=51365.
Augmentation Holter + ajustement fin — CV patient Patch B :
Exactitude : 0.837
F1 macro : 0.844
Rappel classes 0 à 4 : [0.756 0.899 0.935 0.931 0.784]
Paramètres entraînables pendant le fine-tuning : 51365
Journal mis à jour : /home/mapster/ABOUBAKER/SDIA_PRACTICES/SCHOOL/s3-s4/advanced-ai/USECASE_TP/PRACTICE/deliverables/Jalon3/journal_experiences_jalon3.csv


## 17. Comparer les stratégies avec augmentation

Le premier score mesure l’effet de l’augmentation sans aucun ajustement Patch B. Le second mesure la combinaison de cette augmentation avec l’ajustement fin sur les patients patch du pli. On compare les deux aux stratégies précédentes, sans utiliser le jeu de test.

In [35]:
print(f'Holter sans adaptation : F1 macro={F1_PATCH_ASIS:.3f}')
print(f'Holter + augmentation patch, sans adaptation : F1 macro={F1_AUG:.3f}')
print(f'CNN Patch B entraîné seul : F1 macro={F1_PATCH_ONLY:.3f}')
print(f'Extraction de caractéristiques : F1 macro={F1_EXTRACTION:.3f}')
print(f'Ajustement fin complet : F1 macro={F1_FINETUNE:.3f}')
print(f'Augmentation + ajustement fin : F1 macro={F1_AUG_FINETUNE:.3f}')

Holter sans adaptation : F1 macro=0.621
Holter + augmentation patch, sans adaptation : F1 macro=0.687
CNN Patch B entraîné seul : F1 macro=0.344
Extraction de caractéristiques : F1 macro=0.771
Ajustement fin complet : F1 macro=0.823
Augmentation + ajustement fin : F1 macro=0.844


## 18. Décrire le décalage entre Holter et Patch B

On compare ici uniquement les patients d’entraînement des deux appareils. Les mesures brutes décrivent amplitude, variations rapides et énergie fréquentielle. La normalisation par fenêtre devrait égaliser moyenne et écart-type ; elle ne supprime pas les dérives lentes ni les bruits qui changent la forme ou le spectre.

In [37]:
from scipy.signal import welch

def descripteurs_appareil(X):
    # X contient des fenêtres brutes, en mV, de forme (fenêtres, temps).
    X = X.astype('float32')
    frequence, puissance = welch(X, fs=FS, axis=1, nperseg=512)
    puissance_totale = puissance.sum(axis=1) + 1e-12

    def part_bande(min_hz, max_hz):
        masque = (frequence >= min_hz) & (frequence < max_hz)
        return puissance[:, masque].sum(axis=1) / puissance_totale

    return {
        'écart-type brut médian (mV)': float(np.median(X.std(axis=1))),
        'amplitude crête-à-crête médiane (mV)': float(np.median(np.ptp(X, axis=1))),
        'variation absolue moyenne par point (mV)': float(np.median(np.abs(np.diff(X, axis=1)).mean(axis=1))),
        'part puissance 20–62,5 Hz': float(np.median(part_bande(20, FS/2 + 0.01))),
        'part puissance autour de 50 Hz': float(np.median(part_bande(49, 51))),
    }

resume_A = descripteurs_appareil(A['X_train'])
resume_B = descripteurs_appareil(B['X_train'])
print('Descripteurs médians sur les patients train uniquement :')
for nom in resume_A:
    print(f'{nom} — Holter A : {resume_A[nom]:.4f} ; Patch B : {resume_B[nom]:.4f}')

# Vérifier ce que la normalisation par fenêtre retire effectivement.
print('Après z-score, écart-type médian — Holter :',
      round(float(np.median(SA['train'].std(axis=1))), 4),
      '; Patch :', round(float(np.median(SB['train'].std(axis=1))), 4))
print('Après z-score, moyenne absolue médiane — Holter :',
      round(float(np.median(np.abs(SA['train'].mean(axis=1)))), 4),
      '; Patch :', round(float(np.median(np.abs(SB['train'].mean(axis=1)))), 4))

Descripteurs médians sur les patients train uniquement :
écart-type brut médian (mV) — Holter A : 0.2847 ; Patch B : 0.2451
amplitude crête-à-crête médiane (mV) — Holter A : 2.0259 ; Patch B : 1.4470
variation absolue moyenne par point (mV) — Holter A : 0.0763 ; Patch B : 0.0649
part puissance 20–62,5 Hz — Holter A : 0.0486 ; Patch B : 0.0458
part puissance autour de 50 Hz — Holter A : 0.0034 ; Patch B : 0.0032
Après z-score, écart-type médian — Holter : 1.0 ; Patch : 1.0
Après z-score, moyenne absolue médiane — Holter : 0.0 ; Patch : 0.0


## 19. Modèle multitâche : rythme et fréquence cardiaque

Le CNN partage ses couches de détection, puis a deux sorties : cinq classes de rythme et une estimation numérique de la fréquence cardiaque. On compare λ = 0, 0,1 et 1 : λ règle l’importance de l’erreur de fréquence cardiaque dans l’apprentissage. La fréquence cardiaque est standardisée à partir des seuls patients d’entraînement de chaque pli ; les métriques de rythme restent la base du choix. Chaque pli garde trois patients patch à l’écart.

In [39]:
# Construire un CNN à deux sorties en recopiant les poids Holter pour la partie rythme.
def cnn1d_multitache():
    base = cnn1d()
    representation = base.get_layer('representation').output
    fc = layers.Dense(1, name='frequence_cardiaque')(representation)
    return keras.Model(base.input, [base.get_layer('rythme').output, fc],
                       name='cnn1d_multitache')

def recopier_poids_holter(modele):
    # Keras peut ajouter des suffixes aux noms ; copier les couches pondérées dans l’ordre du graphe.
    source = [c for c in cnn_A.layers if c.get_weights()]
    cible = [c for c in modele.layers if c.get_weights()]
    for couche_base, couche_cible in zip(source, cible[:len(source)]):
        assert [w.shape for w in couche_base.get_weights()] == [w.shape for w in couche_cible.get_weights()]
        couche_cible.set_weights(couche_base.get_weights())

LAMBDA_FC = [0.0, 0.1, 1.0]
RESULTATS_LAMBDA = {}
MAE_FC_LAMBDA = {}

for lambda_fc in LAMBDA_FC:
    pred_oof = np.full(len(B['classe_train']), -1, dtype=int)
    pred_fc_oof = np.full(len(B['classe_train']), np.nan, dtype='float32')
    scores_plis = []
    for numero, (idx_train, idx_val) in enumerate(plis_patch, start=1):
        keras.utils.set_random_seed(GRAINE + 500 + int(lambda_fc * 100) + numero)
        modele_multi = cnn1d_multitache()
        recopier_poids_holter(modele_multi)
        for couche in modele_multi.layers:
            couche.trainable = True
        modele_multi.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-4),
            loss=['sparse_categorical_crossentropy', 'mse'],
            loss_weights=[1.0, lambda_fc])

        y_rythme = B['classe_train'][idx_train].astype(int)
        moyenne_fc = float(B['fc_train'][idx_train].mean())
        ecart_fc = float(B['fc_train'][idx_train].std() + 1e-6)
        y_fc = ((B['fc_train'][idx_train] - moyenne_fc) / ecart_fc).astype('float32')
        classes_vues = np.unique(y_rythme)
        poids_pli = compute_class_weight('balanced', classes=classes_vues, y=y_rythme)
        poids_pli = dict(zip(classes_vues.tolist(), poids_pli.tolist()))
        poids_rythme = np.array([poids_pli[int(c)] for c in y_rythme], dtype='float32')
        modele_multi.fit(SB['train'][idx_train], [y_rythme, y_fc],
            sample_weight=[poids_rythme, np.ones(len(idx_train), dtype='float32')],
            epochs=12, batch_size=64, verbose=0)

        proba, pred_fc_standardisee = modele_multi(SB['train'][idx_val], training=False)
        pred_oof[idx_val] = proba.numpy().argmax(axis=1)
        pred_fc_oof[idx_val] = (pred_fc_standardisee.numpy().ravel() * ecart_fc + moyenne_fc)
        f1_pli = f1_score(B['classe_train'][idx_val], pred_oof[idx_val],
                          average='macro', zero_division=0)
        scores_plis.append(f1_pli)
        print(f'λ={lambda_fc:g}, pli {numero} : F1 macro={f1_pli:.3f}.')
        del modele_multi

    f1_multi = f1_score(B['classe_train'], pred_oof, average='macro', zero_division=0)
    acc_multi = accuracy_score(B['classe_train'], pred_oof)
    rappels_multi = recall_score(B['classe_train'], pred_oof, labels=CLASSES,
                                 average=None, zero_division=0)
    mae_fc = float(np.mean(np.abs(B['fc_train'] - pred_fc_oof)))
    RESULTATS_LAMBDA[lambda_fc] = {'f1':float(f1_multi),'accuracy':float(acc_multi),
                                  'recalls':rappels_multi,'folds':scores_plis}
    MAE_FC_LAMBDA[lambda_fc] = mae_fc
    print(f'λ={lambda_fc:g} : exactitude={acc_multi:.3f}, F1 macro={f1_multi:.3f}, '
          f'rappels={np.round(rappels_multi,3)}, MAE fréquence cardiaque={mae_fc:.1f} bpm.')

# Journaliser chaque valeur de lambda sans multiplier les lignes lors d’une réexécution.
with JOURNAL.open(encoding='utf-8-sig', newline='') as f:
    lignes = list(csv.DictReader(f))
for lambda_fc, resultat in RESULTATS_LAMBDA.items():
    nom_exp = f'multitache_lambda_{lambda_fc:g}'
    lignes = [r for r in lignes if r['experience'] != nom_exp]
    lignes.append({'jalon':3, 'experience':nom_exp, 'graine':GRAINE,
        'f1_macro_validation':round(resultat['f1'],4),
        'exactitude_validation':round(resultat['accuracy'],4),
        'rappel_classes_0_a_4':' / '.join(f'{v:.3f}' for v in resultat['recalls']),
        'statut':'réalisé',
        'notes':f'CV 4 plis groupés par patient ; CNN Holter multitâche rythme + fréquence cardiaque ; lambda={lambda_fc:g} ; MAE FC={MAE_FC_LAMBDA[lambda_fc]:.2f} bpm'})
with JOURNAL.open('w', encoding='utf-8-sig', newline='') as f:
    ecrivain=csv.DictWriter(f, fieldnames=champs);ecrivain.writeheader();ecrivain.writerows(lignes)

# Choix provisoire par F1 macro CV : aucun score Patch B test n’est encore consulté ici.
lambda_retenu = max(LAMBDA_FC, key=lambda v: RESULTATS_LAMBDA[v]['f1'])
meilleur_multi = RESULTATS_LAMBDA[lambda_retenu]['f1']
if meilleur_multi > F1_AUG_FINETUNE:
    STRATEGIE_FINALE = 'multitache'
    LAMBDA_FINAL = lambda_retenu
else:
    STRATEGIE_FINALE = 'augmentation_plus_ajustement_fin'
    LAMBDA_FINAL = None
print('Meilleur lambda sur les plis :', lambda_retenu, ' ; F1 macro :', round(meilleur_multi,3))
print('Stratégie retenue avant test :', STRATEGIE_FINALE,
      '(F1 macro CV=', round(max(meilleur_multi,F1_AUG_FINETUNE),3), ')')

λ=0, pli 1 : F1 macro=0.777.
λ=0, pli 2 : F1 macro=0.647.
λ=0, pli 3 : F1 macro=0.853.
λ=0, pli 4 : F1 macro=0.509.
λ=0 : exactitude=0.830, F1 macro=0.828, rappels=[0.756 0.87  0.935 0.862 0.838], MAE fréquence cardiaque=43.2 bpm.
λ=0.1, pli 1 : F1 macro=0.745.
λ=0.1, pli 2 : F1 macro=0.654.
λ=0.1, pli 3 : F1 macro=0.859.
λ=0.1, pli 4 : F1 macro=0.507.
λ=0.1 : exactitude=0.833, F1 macro=0.832, rappels=[0.748 0.884 0.935 0.828 0.892], MAE fréquence cardiaque=18.6 bpm.
λ=1, pli 1 : F1 macro=0.625.
λ=1, pli 2 : F1 macro=0.622.
λ=1, pli 3 : F1 macro=0.837.
λ=1, pli 4 : F1 macro=0.491.
λ=1 : exactitude=0.810, F1 macro=0.801, rappels=[0.782 0.841 0.935 0.828 0.676], MAE fréquence cardiaque=15.8 bpm.
Meilleur lambda sur les plis : 0.1  ; F1 macro : 0.832
Stratégie retenue avant test : augmentation_plus_ajustement_fin (F1 macro CV= 0.844 )


## 20. Vérifier les propriétés de la normalisation et les paramètres

Pour un gain positif a et un décalage constant b, z(a·x+b) = z(x) : le z-score retire ces deux changements fenêtre par fenêtre. Un bruit qui varie dans le temps n’est pas un simple décalage ni un gain et peut donc rester après normalisation. Pour l’extraction de caractéristiques, seule la sortie rythme est entraînable ; pour l’ajustement fin, toutes les couches le sont.

In [41]:
# Vérification numérique : appliquer un gain positif et un décalage constant avant le z-score.
echantillon = A['X_train'][:1].astype('float32')
z_original = normaliser_fenetres(echantillon)
z_gain_decalage = normaliser_fenetres(2.5 * echantillon + 3.0)
erreur_gain_decalage = float(np.max(np.abs(z_original - z_gain_decalage)))
print('Écart maximal après gain=2,5 et décalage=3 mV :', f'{erreur_gain_decalage:.2e}')
assert erreur_gain_decalage < 1e-4

# Le bruit temporel n’est pas annulé par cette normalisation.
rng_verif = np.random.default_rng(GRAINE + 900)
echantillon_bruite = echantillon + rng_verif.normal(0,0.03,echantillon.shape).astype('float32')
erreur_bruit = float(np.mean(np.abs(z_original - normaliser_fenetres(echantillon_bruite))))
print('Écart moyen après ajout de bruit variable :', f'{erreur_bruit:.4f}')

# Recompter réellement les paramètres ajustables dans les deux configurations.
modele_compte = keras.models.clone_model(cnn_A)
modele_compte.set_weights(cnn_A.get_weights())
for couche in modele_compte.layers: couche.trainable = False
modele_compte.get_layer('rythme').trainable = True
PARAM_EXTRACTION_VERIFIES = int(sum(np.prod(w.shape) for w in modele_compte.trainable_weights))
for couche in modele_compte.layers: couche.trainable = True
PARAM_FINETUNE_VERIFIES = int(sum(np.prod(w.shape) for w in modele_compte.trainable_weights))
print('Paramètres entraînables — extraction :', PARAM_EXTRACTION_VERIFIES)
print('Paramètres entraînables — ajustement fin :', PARAM_FINETUNE_VERIFIES)
assert PARAM_EXTRACTION_VERIFIES == 325
assert PARAM_FINETUNE_VERIFIES == 51365

Écart maximal après gain=2,5 et décalage=3 mV : 1.43e-05
Écart moyen après ajout de bruit variable : 0.0899
Paramètres entraînables — extraction : 325
Paramètres entraînables — ajustement fin : 51365


## 21. Entraîner le modèle retenu, puis évaluer le test une seule fois

Le choix est maintenant figé à partir de la validation croisée par patient. On réentraîne cette stratégie sur les 12 patients Patch B train, puis seulement maintenant on charge le test. Après ce calcul, aucun réglage ne sera fait à partir du score test.

In [43]:
# Réentraîner sur tous les patients patch train selon la stratégie choisie avant d’ouvrir le test.
if STRATEGIE_FINALE == 'multitache':
    modele_final = cnn1d_multitache()
    recopier_poids_holter(modele_final)
    for couche in modele_final.layers: couche.trainable = True
    moyenne_fc_final = float(B['fc_train'].mean())
    ecart_fc_final = float(B['fc_train'].std() + 1e-6)
    y_fc_final = ((B['fc_train'] - moyenne_fc_final) / ecart_fc_final).astype('float32')
    classes_vues = np.unique(B['classe_train'])
    poids_final = compute_class_weight('balanced', classes=classes_vues, y=B['classe_train'])
    poids_final = dict(zip(classes_vues.tolist(), poids_final.tolist()))
    poids_rythme_final = np.array([poids_final[int(c)] for c in B['classe_train']],dtype='float32')
    modele_final.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-4),
        loss=['sparse_categorical_crossentropy','mse'],
        loss_weights=[1.0,LAMBDA_FINAL])
    modele_final.fit(SB['train'],[B['classe_train'].astype(int),y_fc_final],
        sample_weight=[poids_rythme_final,np.ones(len(y_fc_final),dtype='float32')],
        epochs=12,batch_size=64,verbose=0)
else:
    modele_final = keras.models.clone_model(cnn_A_aug)
    modele_final.set_weights(cnn_A_aug.get_weights())
    for couche in modele_final.layers: couche.trainable = True
    classes_vues = np.unique(B['classe_train'])
    poids_final = compute_class_weight('balanced',classes=classes_vues,y=B['classe_train'])
    poids_final = dict(zip(classes_vues.tolist(),poids_final.tolist()))
    modele_final.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-4),
                         loss='sparse_categorical_crossentropy')
    modele_final.fit(SB['train'],B['classe_train'].astype(int),class_weight=poids_final,
                     epochs=12,batch_size=64,verbose=0)

# Charger maintenant les données test, faire une seule prédiction et calculer les métriques finales.
with np.load(PRACTICE_DIR / 'data' / 'patch_B' / 'uc3_patchB.npz') as f:
    X_patch_test_brut=f['X_test'].copy()
    y_patch_test=f['classe_test'].copy().astype(int)
    patients_patch_test=f['patient_test'].copy()
X_patch_test=normaliser_fenetres(X_patch_test_brut)
if STRATEGIE_FINALE == 'multitache':
    sorties_test=modele_final(X_patch_test,training=False)
    proba_test=sorties_test[0].numpy()
else:
    proba_test=modele_final(X_patch_test,training=False).numpy()
pred_test=proba_test.argmax(axis=1)
F1_TEST_FINAL=f1_score(y_patch_test,pred_test,average='macro',zero_division=0)
ACC_TEST_FINAL=accuracy_score(y_patch_test,pred_test)
RAPPELS_TEST_FINAL=recall_score(y_patch_test,pred_test,labels=CLASSES,average=None,zero_division=0)
print('ÉVALUATION FINALE — Patch B test, une seule fois')
print('Stratégie retenue avant test :',STRATEGIE_FINALE, 'lambda=',LAMBDA_FINAL)
print('Patients test :',len(np.unique(patients_patch_test)))
print('Exactitude test :',round(ACC_TEST_FINAL,3))
print('F1 macro test :',round(F1_TEST_FINAL,3))
print('Rappel par classe test :',np.round(RAPPELS_TEST_FINAL,3))
modele_final.save(DELIVERABLE_DIR / 'modele_final_jalon3.keras')

with JOURNAL.open(encoding='utf-8-sig',newline='') as f:
    lignes=list(csv.DictReader(f))
colonnes_test=['f1_macro_test','exactitude_test','rappel_classes_test']
champs_complets=list(champs)+[c for c in colonnes_test if c not in champs]
lignes=[r for r in lignes if r['experience']!='evaluation_finale_test_unique']
lignes.append({'jalon':3,'experience':'evaluation_finale_test_unique','graine':GRAINE,
    'f1_macro_validation':'','exactitude_validation':'','rappel_classes_0_a_4':'',
    'f1_macro_test':round(float(F1_TEST_FINAL),4),
    'exactitude_test':round(float(ACC_TEST_FINAL),4),
    'rappel_classes_test':' / '.join(f'{v:.3f}' for v in RAPPELS_TEST_FINAL),
    'statut':'test final, une seule évaluation',
    'notes':f'stratégie choisie par CV patient avant ouverture du test : {STRATEGIE_FINALE} ; aucune modification après ce score'})
with JOURNAL.open('w',encoding='utf-8-sig',newline='') as f:
    ecrivain=csv.DictWriter(f,fieldnames=champs_complets);ecrivain.writeheader();ecrivain.writerows(lignes)

ÉVALUATION FINALE — Patch B test, une seule fois
Stratégie retenue avant test : augmentation_plus_ajustement_fin lambda= None
Patients test : 42
Exactitude test : 0.87
F1 macro test : 0.803
Rappel par classe test : [0.824 0.854 0.972 1.    0.875]


## Terminé — éléments de synthèse

- Comparer les stratégies patch par validation croisée groupée par patient.
- Expliquer le décalage appareil, les limites du z-score et les paramètres entraînables.
- Étudier le modèle multitâche et λ, choisir avant le test.
- Évaluer une seule fois le modèle retenu sur le test et documenter les limites des données simulées.
